In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import comb, gammaln
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, mean_squared_error, mean_absolute_error, r2_score, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import warnings
warnings.filterwarnings("ignore")

try:
    import mord
    has_mord = True
except ImportError:
    has_mord = False
    print("⚠️ mord (ordinal logistic regression) not installed; skipping this model.")

# ==========================================================
# 1. 数据读取与划分
# ==========================================================
data = pd.read_csv("StressLevelDataset.csv")
X = data.drop(columns=["stress_level"]).values
y = data["stress_level"].astype(int).values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
Z = np.column_stack([np.ones(X_scaled.shape[0]), X_scaled])
Z_train, Z_test, y_train, y_test = train_test_split(Z, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_test = Z_train[:, 1:], Z_test[:, 1:]  # for sklearn models
print(f"Training set: {Z_train.shape}, Test set: {Z_test.shape}")

# ==========================================================
# 2. Bernstein-Binomial & Beta-Binomial 类定义（简化版）
# ==========================================================
class BernsteinBinomialRegression:
    def __init__(self, K=3, m=2):
        self.K = K; self.m = m
        self.alpha = None; self.phi = None; self.beta = None
        self.log_likelihood = None; self.n_params = None

    def _compute_eta(self, Z, alpha, phi, beta):
        n = Z.shape[0]; Z_beta = Z @ beta
        eta = np.zeros((n, self.K + 1))
        eta[:, 0] = Z_beta
        for k in range(1, self.K + 1):
            eta[:, k] = alpha[k-1] + phi[k-1]*Z_beta
        return eta

    def _compute_lambda(self, eta):
        max_eta = np.max(eta, axis=1, keepdims=True)
        exp_eta = np.exp(eta - max_eta)
        return exp_eta / np.sum(exp_eta, axis=1, keepdims=True)

    def _bernstein_basis(self, p, y_obs):
        return comb(self.m, y_obs) * (p**y_obs) * ((1-p)**(self.m - y_obs))

    def _log_likelihood(self, params, Z, y):
        n, p = Z.shape; K = self.K
        alpha = params[:K]; phi = np.exp(params[K:2*K]); beta = params[2*K:2*K+p]
        eta = self._compute_eta(Z, alpha, phi, beta)
        lambda_mat = self._compute_lambda(eta)
        logL = 0.0
        for i in range(n):
            mix = 0.0
            for k in range(K+1):
                mix += lambda_mat[i,k]*self._bernstein_basis(k/K, y[i])
            logL += np.log(max(mix, 1e-12))
        return logL

    def fit(self, Z, y, max_iter=1000):
        n, p = Z.shape; K = self.K
        np.random.seed(42)
        alpha_init = np.random.normal(0,0.1,K)
        phi_init = np.log(np.ones(K))
        beta_init = np.random.normal(0,0.1,p)
        init_params = np.concatenate([alpha_init, phi_init, beta_init])
        bounds = [(-5,5)]*len(init_params)
        result = minimize(lambda par: -self._log_likelihood(par,Z,y),
                          init_params, method="L-BFGS-B", bounds=bounds,
                          options={'maxiter':max_iter})
        self.alpha = result.x[:K]; self.phi = np.exp(result.x[K:2*K])
        self.beta = result.x[2*K:2*K+p]; self.log_likelihood=self._log_likelihood(result.x,Z,y)
        self.n_params=len(result.x); self.optimization_result=result
        return self

    def predict_proba(self,Z):
        eta=self._compute_eta(Z,self.alpha,self.phi,self.beta)
        lam=self._compute_lambda(eta)
        prob=np.zeros((Z.shape[0],self.m+1))
        for i in range(Z.shape[0]):
            for y in range(self.m+1):
                prob[i,y]=sum(lam[i,k]*self._bernstein_basis(k/self.K,y) for k in range(self.K+1))
        return prob/np.sum(prob,axis=1,keepdims=True)

    def predict(self,Z):
        return np.argmax(self.predict_proba(Z),axis=1)

class BetaBinomialRegression:
    def __init__(self, m=2): self.m=m; self.coeff=None; self.log_likelihood=None; self.n_params=None
    def _log_likelihood(self, params, Z, y):
        logit=Z@params; p=1/(1+np.exp(-logit)); p=np.clip(p,1e-5,1-1e-5)
        disp=5; alpha=p*disp; beta=(1-p)*disp; logL=0
        for i in range(len(y)):
            logL += (gammaln(self.m+1)-gammaln(y[i]+1)-gammaln(self.m-y[i]+1)
                     +gammaln(y[i]+alpha[i])+gammaln(self.m-y[i]+beta[i])
                     -gammaln(self.m+alpha[i]+beta[i])
                     +gammaln(alpha[i]+beta[i])-gammaln(alpha[i])-gammaln(beta[i]))
        return logL
    def fit(self,Z,y):
        res=minimize(lambda p:-self._log_likelihood(p,Z,y),np.zeros(Z.shape[1]),method="L-BFGS-B")
        self.coeff=res.x; self.log_likelihood=self._log_likelihood(res.x,Z,y); self.n_params=Z.shape[1]
        return self
    def predict_proba(self,Z):
        logit=Z@self.coeff; p=1/(1+np.exp(-logit))
        return np.column_stack([(1-p)**2,2*p*(1-p),p**2])
    def predict(self,Z): return np.argmax(self.predict_proba(Z),axis=1)

# ==========================================================
# 3. 重新训练最优K=6的 Bernstein-Binomial 和 Beta-Binomial
# ==========================================================
K_star=6; m=2
bern_model=BernsteinBinomialRegression(K=K_star,m=m).fit(Z_train,y_train)
beta_model=BetaBinomialRegression(m=m).fit(Z_train,y_train)

# ==========================================================
# 4. 其他机器学习模型（多项logit、有序logit、随机森林、SVM）
# ==========================================================
models = {
    "Bernstein-Binomial": bern_model,
    "Beta-Binomial": beta_model,
    "Multinomial Logistic": LogisticRegression(multi_class="multinomial", max_iter=1000).fit(X_train,y_train),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train,y_train),
    "SVM (RBF)": SVC(kernel="rbf", probability=True, random_state=42).fit(X_train,y_train)
}
if has_mord:
    models["Ordinal Logistic"] = mord.LogisticIT().fit(X_train, y_train)

# ==========================================================
# 5. 统一评估函数
# ==========================================================
def evaluate_model(name, model, Z_train, y_train, Z_test, y_test):
    # 预测与概率
    if hasattr(model, "predict_proba"):
        prob_test = model.predict_proba(Z_test if "Binomial" in name else X_test)
    else:
        prob_test = None
    y_pred = model.predict(Z_test if "Binomial" in name else X_test)
    
    # 性能指标
    acc = accuracy_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Log-likelihood & AIC
    if prob_test is not None:
        eps = 1e-12
        logL = np.sum(np.log(np.clip(prob_test[np.arange(len(y_test)), y_test], eps, 1)))
        n_params = getattr(model, "n_params", None) or prob_test.shape[1]
        aic = 2*n_params - 2*logL
    else:
        logL, aic = np.nan, np.nan

    return {"Model": name, "Accuracy": acc, "MSE": mse, "MAE": mae, "R2": r2, "LogLik": logL, "AIC": aic}

# ==========================================================
# 6. 在测试集上统一评估
# ==========================================================
results=[]
for name, model in models.items():
    res=evaluate_model(name, model, Z_train, y_train, Z_test, y_test)
    results.append(res)
comparison_df=pd.DataFrame(results)
comparison_df.sort_values("Accuracy",ascending=False,inplace=True)
print("\n=================== COMPREHENSIVE MODEL COMPARISON ===================")
print(comparison_df.round(4))
